# Topic: Transfer Learning & Fine-Tuning in TensorFlow

## Definition (30-second explanation)
- Transfer learning is the process of taking a neural network trained on a large dataset (like ImageNet) and adapting it to a different but related task.
- It involves keeping the pre-trained weights to extract useful features (like edges or shapes) and replacing the final layers (the classification head) to output predictions for the new specific classes.

## Why Interviewers Ask This
- **Industry Reality:** Almost no one trains massive models (ResNet, BERT) from scratch; fine-tuning is the standard industry workflow.
- **Troubleshooting:** Tests if you know how to avoid "catastrophic forgetting" (destroying pre-trained weights).
- **Resource Management:** Shows you understand how to achieve high accuracy with limited data and compute constraints.

## Core Concepts
- **Freezing Base Layers:** Setting `trainable = False` on the imported model to prevent its weights from updating during backpropagation.
- **Replacing the Head:** Removing the original output layer (e.g., 1000 classes) and adding custom Dense layers (e.g., 2 classes for Cat vs. Dog).
- **Feature Extraction:** Only training the new classification head while the base model acts as a static feature extractor.
- **Fine-Tuning:** Unfreezing the top layers of the base model and training them jointly with the new head using a very low learning rate.

## When to Use
- When you have a small to medium-sized dataset that would overfit a deep network trained from scratch.
- When your target domain is visually or semantically similar to the source dataset.
- When computational resources or time constraints limit full model training.

## Advantages
- **Faster Convergence:** The model already knows how to process basic features.
- **Requires Less Data:** Mitigates overfitting on small target datasets.
- **Higher Baseline Accuracy:** Reaps the benefits of architectures trained on millions of examples.

## Limitations
- **Domain Mismatch:** Performs poorly if target data is radically different (e.g., natural images vs. satellite/medical imagery).
- **Rigid Architectures:** You are often locked into the input image size and channel constraints of the pre-trained model.
- **Batch Normalization Quirks:** Frozen BatchNormalization layers can cause silent bugs during fine-tuning if not handled carefully.

## Common Comparisons
- **Feature Extraction vs. Fine-Tuning:** Feature extraction freezes the entire base (fast, safe). Fine-tuning unfreezes parts of the base (higher accuracy, risks overfitting/catastrophic forgetting).
- **Shallow Fine-Tuning vs. Deep Fine-Tuning:** Shallow unfreezes only the last few conv blocks; Deep unfreezes everything.

## Common Interview Traps
- **Trap 1:** Unfreezing the base model immediately. *Correction:* Always train the new random head *first* while the base is frozen; otherwise, random gradients will destroy the pre-trained weights.
- **Trap 2:** Using the same learning rate for fine-tuning. *Correction:* Use a much smaller learning rate (e.g., 1e-5) when unfreezing base layers.
- **Trap 3:** Forgetting to preprocess inputs. *Correction:* Always use the specific `preprocess_input` function associated with the chosen model (e.g., `tf.keras.applications.resnet.preprocess_input`).

## Python / SQL Syntax (if applicable)
```python
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

# 1. Feature Extraction setup
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze the base

x = GlobalAveragePooling2D()(base_model.output)
output = Dense(10, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=output)

model.compile(optimizer='adam', loss='categorical_crossentropy')
# -> Train the model here (Head only)

# 2. Fine-Tuning setup (after head converges)
base_model.trainable = True
for layer in base_model.layers[:100]: # Freeze bottom 100 layers
    layer.trainable = False

# Must recompile after changing trainable status, use low LR!
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy')
# -> Train again (Fine-tuning)
```

## 45-Second Interview Answer
"Transfer learning utilizes representations from large datasets to solve new tasks with less data. My standard workflow involves two steps. First, Feature Extraction: I import a base model like ResNet without its top layers, freeze its weights, and attach a new classification head. I train this head until convergence. Second, Fine-Tuning: I unfreeze the top layers of the base model and resume training with a much lower learning rate. This two-step process adapts the pre-trained features to my specific dataset without suffering from catastrophic forgetting."

## Practice Questions:

### Q1:
**Scenario:** 
You are tasked with fine-tuning a `MobileNetV2` model for a binary classification task (cats vs. dogs) for a mobile app deployment. You have already completed the "Feature Extraction" phase where the base was completely frozen and the head was trained. Now, you need to unfreeze the model to do full Fine-Tuning.

**Dummy Data / Setup Code:**
```python
import tensorflow as tf
import numpy as np

# Dummy dataset: 100 images of 160x160x3, binary labels
X_train = np.random.rand(100, 160, 160, 3).astype(np.float32)
y_train = np.random.randint(0, 2, 100)

# The model as it exists after Feature Extraction Phase
base_model = tf.keras.applications.MobileNetV2(input_shape=(160, 160, 3), include_top=False, weights='imagenet')
base_model.trainable = False 
inputs = tf.keras.Input(shape=(160, 160, 3))
x = base_model(inputs, training=False) 
x = tf.keras.layers.GlobalAveragePooling2D()(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model = tf.keras.Model(inputs, outputs)

# Assume model.fit(...) has already been run for 10 epochs here.
```

**Interview Question:**
"Looking at the current state of the model above, write the TensorFlow Python code to transition into the Fine-Tuning phase. Specifically, unfreeze the base model starting from layer 100 onwards. Explain your choice of learning rate, and tell me: why is there a potential trap regarding BatchNormalization layers here, and how did the provided setup code already mitigate it?"

In [1]:
import tensorflow as tf
import numpy as np

# ==========================================
# 1. SETUP & DUMMY DATA
# ==========================================
# 100 dummy images (160x160 RGB) and binary labels
X_train = np.random.rand(100, 160, 160, 3).astype(np.float32)
y_train = np.random.randint(0, 2, 100)

# ==========================================
# 2. PHASE 1: FEATURE EXTRACTION
# ==========================================
# Load pre-trained MobileNetV2 without the classification head
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3), 
    include_top=False, 
    weights='imagenet'
)

# Freeze the entire base model so its weights don't update
base_model.trainable = False

# Build the new model architecture
inputs = tf.keras.Input(shape=(160, 160, 3))

# CRITICAL INTERVIEW POINT: 
# passing training=False ensures BatchNormalization layers run in 
# inference mode. They will use frozen ImageNet statistics rather 
# than recalculating them on our small target batches.
x = base_model(inputs, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model = tf.keras.Model(inputs, outputs)

# Compile with a standard learning rate (e.g., 1e-3) 
# because we are training a randomly initialized head from scratch.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Starting Feature Extraction Training...")
# Train only the top Dense layer until it converges
model.fit(X_train, y_train, epochs=5, batch_size=16)


# ==========================================
# 3. PHASE 2: FINE-TUNING
# ==========================================
# Unfreeze the base model
base_model.trainable = True

# Re-freeze the bottom 100 layers (keep low-level feature extractors fixed)
for layer in base_model.layers[:100]:
    layer.trainable = False

# CRITICAL INTERVIEW POINT (The BN Trap Fix): 
# Explicitly freeze BatchNormalization layers. Even if they are past 
# layer 100, we don't want our small target batch sizes ruining the 
# pre-calculated running means and variances.
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Recompile with a MUCH LOWER learning rate (e.g., 1e-5)
# This prevents destroying the pre-trained weights with massive gradient updates.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nStarting Fine-Tuning...")
# Continue training (updates the top unfrozen conv layers and the dense head)
model.fit(X_train, y_train, epochs=10, batch_size=16)

I0000 00:00:1788841585.730118   13408 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788841585.780293   13408 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788841589.172400   13408 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788841594.838173   13408 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9507 MB mem

Starting Feature Extraction Training...
Epoch 1/5


I0000 00:00:1788841598.240065   13615 service.cc:153] XLA service 0x7f0e8405da00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1788841598.240114   13615 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4080 Laptop GPU, Compute Capability 8.9 (Driver: 13.4.0; Runtime: 12.1.0; Toolkit: 12.5.0; DNN: 9.25.1)
I0000 00:00:1788841598.321743   13615 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1788841599.124574   13615 cuda_dnn.cc:461] Loaded cuDNN version 92501
E0000 00:00:1788841601.985796   13615 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


5/7 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.5000 - loss: 0.7497

I0000 00:00:1788841607.664883   13615 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
E0000 00:00:1788841609.582484   13615 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


7/7 ━━━━━━━━━━━━━━━━━━━━ 17s 1s/step - accuracy: 0.5400 - loss: 0.7384
Epoch 2/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5000 - loss: 0.7447
Epoch 3/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5000 - loss: 0.7108
Epoch 4/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5400 - loss: 0.7221
Epoch 5/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.4700 - loss: 0.6991

Starting Fine-Tuning...
Epoch 1/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 15s 479ms/step - accuracy: 0.4500 - loss: 0.8204
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.4900 - loss: 0.7575
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5400 - loss: 0.7177
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6300 - loss: 0.6593
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5200 - loss: 0.6841
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5700 - loss: 0.6547
Epoch 7/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5600 - loss: 

### Q2: Domain Mismatch in Transfer Learning

**Question:** 
Suppose you fine-tune an ImageNet-trained model for detecting MRI tumors. Even with a tiny learning rate, it overfits and fails to generalize. Why does this happen, and how do you fix it?

**Answer:**
"This is a classic case of severe **Domain Mismatch**. ImageNet pre-training teaches the model to look for macroscopic, structural features like animal ears or car wheels. Medical imaging, however, is a fine-grained anomaly detection task that relies on microscopic textural differences. The pre-trained weights are essentially useless—and sometimes actively harmful—for MRIs, leading the model to overfit on the small target dataset rather than learning generalizable features. 

To fix this, I would change my strategy in one of two ways:
1. **Domain-Specific Weights:** I would swap ImageNet weights for weights pre-trained on medical data (like RadImageNet).
2. **Train from Scratch / Shallow Extraction:** If I only have ImageNet, I would either discard it and train a smaller CNN entirely from scratch (adding heavy data augmentation), or I would freeze only the first 1-2 convolutional blocks (which detect basic edges) and train a much deeper custom network on top."

**Common Mistakes Candidates Make:**
- Suggesting "more dropout" or "L2 regularization". While those prevent overfitting, they don't solve the root cause, which is that the base weights are semantically incompatible with the target data.

**One Likely Interviewer Follow-up:**
- *Interviewer:* "If you train from scratch on the MRI data, what challenges will you face and how will you overcome them?"
- *Answer:* "Medical datasets are usually very small and highly imbalanced (many healthy scans, few tumors). I'd use aggressive data augmentation (rotations, elastic deformations), focal loss to handle the class imbalance, and potentially k-fold cross-validation to ensure reliable evaluation."

### Q3:
- Using the train_ds and val_ds generated below, write the TensorFlow code to:

1. Import MobileNetV2 and its specific preprocess_input function.

2. Build the model. Apply the preprocess_input function inside the model logic (e.g., wrap it in a tf.keras.layers.Lambda layer right after your Inputs).

3. Execute Feature Extraction for 3 epochs on train_ds (validating on val_ds).

4. Unfreeze only the top 20 layers of the base model, explicitly handle the Batch Normalization trap, and execute Fine-Tuning for another 3 epochs.

In [7]:
import tensorflow as tf
import numpy as np

# Generate dummy data that mimics raw CIFAR-10 (0-255 pixel values, uint8)
X_train= np.random.randint(0, 256, size= (500, 32, 32, 3), dtype= np.uint8)
y_train= np.random.randint(0, 2, 500) # Binary Labels

X_test= np.random.randint(0, 256, (100, 32, 32, 3), dtype= np.uint8)
y_test= np.random.randint(0, 2, 100)

# Function to Resize Images:
def prepare_data(image, label):
    # Resizing Image to MobileNetV2 Minimum Size:
    image= tf.image.resize(images= image, size= (96, 96))
    return image, label

# Creating TF Data Pipeline:
train_ds= tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds= tf.data.Dataset.from_tensor_slices((X_test, y_test))

train_ds = train_ds.map(prepare_data).batch(32).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(prepare_data).batch(32).prefetch(tf.data.AUTOTUNE)

print('Data Ready!!')

Data Ready!!


In [13]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Lambda
from tensorflow.keras import Input
from tensorflow.keras.models import Model

# ==========================================
# FEATURE EXTRACTION
# ==========================================
base_model = MobileNetV2(input_shape= (96, 96, 3), weights= 'imagenet', include_top= False)
base_model.trainable = False

inputs = Input(shape= (96, 96, 3))
x = Lambda(preprocess_input)(inputs)

x = base_model(x, training=False) 

x = GlobalAveragePooling2D()(x)

outputs = Dense(1, activation='sigmoid')(x) 
model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(), # Note: BinaryCrossentropy handles metrics better
    metrics=['accuracy']
)

print("Training Head...")
model.fit(train_ds, validation_data= test_ds, epochs=3)


# ==========================================
# FINE-TUNING
# ==========================================
base_model.trainable = True

# FIX 3: Freeze everything up to the last 20 layers ([:-20]), leaving only the top 20 unfrozen
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Freeze Batch Norm Layers globally to be safe
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate= 0.00001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

print("\nFine-Tuning Top 20 Layers...")
model.fit(train_ds, validation_data= test_ds, epochs= 3)

Training Head...
Epoch 1/3


/home/shail/interview-prep/interview_env/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


16/16 ━━━━━━━━━━━━━━━━━━━━ 9s 336ms/step - accuracy: 0.4980 - loss: 0.7689 - val_accuracy: 0.5100 - val_loss: 0.7178
Epoch 2/3
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.4780 - loss: 0.7183 - val_accuracy: 0.4700 - val_loss: 0.7027
Epoch 3/3
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4900 - loss: 0.6937 - val_accuracy: 0.4600 - val_loss: 0.7038

Fine-Tuning Top 20 Layers...
Epoch 1/3
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 168ms/step - accuracy: 0.5340 - loss: 0.7086 - val_accuracy: 0.5000 - val_loss: 0.7100
Epoch 2/3
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.5560 - loss: 0.6807 - val_accuracy: 0.4500 - val_loss: 0.7015
Epoch 3/3
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5980 - loss: 0.6760 - val_accuracy: 0.4600 - val_loss: 0.7022
